This is a simple Script for my workplace to automate a excel related task. This script reads an Excel sales file (input.xls or input.xlsx), identifies the header row containing CUSTOMER PHONE, and loads the data. It cleans and sorts the records by phone number, then flags customers with multiple purchases as “frequent buyers.” The output is a formatted Excel workbook (Customer Wise Sales Report of [date].xlsx) where frequent buyers are highlighted in rotating colors, metadata is preserved, column widths are optimized, and a legend sheet explains the color coding.

In [1]:
pip install pandas openpyxl xlrd

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
import sys
import pandas as pd
import openpyxl
from openpyxl.styles import PatternFill, Font, Alignment
from openpyxl.utils import get_column_letter

HIGHLIGHT_COLORS = [
    'FFF2CC',  # yellow
    'FFD7BE',  # orange
    'D5E8D4',  # green
    'DAE8FC',  # blue
    'E1D5E7',  # purple
    'F8CECC',  # pink
]

COLUMN_WIDTHS = {
    'DATE': 13, 'CUSTOMER CODE': 15, 'CUSTOMER NAME': 24,
    'CUSTOMER PHONE': 16, 'SALESMAN': 13, 'EMPLOYER NAME': 18,
    'INVOICE NO': 17, 'COST VALUE': 12, 'TOTAL': 12,
    'VAT AMT': 11, 'DISC AMT': 11, 'EXG AMT': 11, 'RTN AMT': 11,
    'ADJ AMT': 11, 'NET AMT': 12, 'CASH AMT': 12, 'CARD AMT': 12,
    'PAYMENT TYPE': 14, 'RTN INV REF': 15, 'ADV AMT': 11, 'RDM VAL': 11,
}


def find_input_file(folder):
    for name in ('input.xlsx', 'input.xls'):
        path = os.path.join(folder, name)
        if os.path.isfile(path):
            return path
    return None


def find_header_row(raw_df):
    for i in range(min(30, len(raw_df))):
        row_vals = [str(v).strip().upper() for v in raw_df.iloc[i] if pd.notna(v)]
        if 'CUSTOMER PHONE' in row_vals:
            return i
    return None


def clean_phone(val):
    if pd.isna(val) or str(val).strip() in ('', 'nan'):
        return ''
    try:
        return str(int(float(val)))
    except (ValueError, OverflowError):
        return str(val).strip()


def main():
    try:
        folder = os.path.dirname(os.path.abspath(__file__))
    except NameError:
        folder = os.getcwd() 

    input_path = find_input_file(folder)
    if not input_path:
        print("ERROR: No input file found.")
        print(f"       Place your file in: {folder}")
        print("       and name it:  input.xls  OR  input.xlsx")
        sys.exit(1)

    from datetime import date
    today = (date.today() - __import__('datetime').timedelta(days=1)).strftime('%d-%m-%Y')
    output_path = os.path.join(folder, f'Customer Wise Sales Report of {today}.xlsx')
    print(f"Input  : {input_path}")
    print(f"Output : {output_path}")

    print("\n[1/4] Reading file...")
    ext = os.path.splitext(input_path)[1].lower()
    engine = 'xlrd' if ext == '.xls' else 'openpyxl'

    raw = pd.read_excel(input_path, engine=engine, header=None)
    header_row = find_header_row(raw)

    if header_row is None:
        print("ERROR: Could not find 'CUSTOMER PHONE' column in the file.")
        print("       Make sure the file has the correct column headers.")
        sys.exit(1)

    meta_lines = []
    for i in range(header_row):
        vals = [str(v).strip() for v in raw.iloc[i] if pd.notna(v) and str(v).strip()]
        if vals:
            meta_lines.append(' | '.join(vals))

    df = pd.read_excel(input_path, engine=engine, header=header_row)
    df = df.dropna(how='all').reset_index(drop=True)
    df['CUSTOMER PHONE'] = df['CUSTOMER PHONE'].apply(clean_phone)

    print(f"        {len(df):,} data rows loaded.")


    print("[2/4] Sorting by phone number...")
    df['_sort'] = df['CUSTOMER PHONE'].apply(
        lambda x: x if x and len(x) > 4 else '\xff'
    )
    df = df.sort_values('_sort').drop(columns=['_sort']).reset_index(drop=True)

    print("[3/4] Identifying frequent buyers...")
    valid = df['CUSTOMER PHONE'][df['CUSTOMER PHONE'].str.len() > 4]
    counts = valid.value_counts()
    frequent = set(counts[counts > 1].index)
    color_map = {
        phone: HIGHLIGHT_COLORS[i % len(HIGHLIGHT_COLORS)]
        for i, phone in enumerate(sorted(frequent))
    }
    print(f"        {len(frequent):,} unique phone numbers with multiple purchases.")


    print("[4/4] Writing output.xlsx...")
    wb = openpyxl.Workbook()
    ws = wb.active
    ws.title = "Customer Sales Report"


    for i, line in enumerate(meta_lines, start=1):
        c = ws.cell(row=i, column=1, value=line)
        c.font = Font(bold=(i == 1), size=12 if i == 1 else 10, name='Arial')

    note_row = len(meta_lines) + 1
    ws.cell(row=note_row, column=1,
            value='Sorted by Customer Phone  |  Highlighted = Frequent Buyer'
            ).font = Font(italic=True, size=9, name='Arial', color='666666')

    header_row_excel = note_row + 2

    data_cols = [c for c in df.columns if not c.startswith('Unnamed')]
    col_index = {col: i + 1 for i, col in enumerate(data_cols)}

    for col, num in col_index.items():
        cell = ws.cell(row=header_row_excel, column=num, value=col)
        cell.font = Font(bold=True, name='Arial', color='FFFFFF')
        cell.fill = PatternFill('solid', start_color='1F4E79')
        cell.alignment = Alignment(horizontal='center', wrap_text=True)

    for offset, (_, row) in enumerate(df.iterrows()):
        erow = header_row_excel + 1 + offset
        phone = row.get('CUSTOMER PHONE', '')
        fill = (PatternFill('solid', start_color=color_map[phone])
                if phone in frequent else None)

        for col, num in col_index.items():
            val = row.get(col, '')
            val = '' if pd.isna(val) else val
            cell = ws.cell(row=erow, column=num, value=val)
            cell.font = Font(name='Arial', size=9)
            if fill:
                cell.fill = fill

    for col, num in col_index.items():
        ws.column_dimensions[get_column_letter(num)].width = COLUMN_WIDTHS.get(col, 13)
    ws.freeze_panes = ws.cell(row=header_row_excel + 1, column=1)

    ls = wb.create_sheet("Legend")
    ls['A1'] = 'Color Legend'
    ls['A1'].font = Font(bold=True, size=13, name='Arial')
    ls['A3'] = 'Color';          ls['B3'] = 'Meaning'
    ls['A3'].font = ls['B3'].font = Font(bold=True, name='Arial')
    ls['A4'] = '(no highlight)'; ls['B4'] = 'Customer purchased only once'
    ls['A5'].fill = PatternFill('solid', start_color='FFF2CC')
    ls['A5'] = 'Highlighted (6 rotating colors)'
    ls['B5'] = 'Frequent buyer — same phone number appears more than once'
    ls['A7'] = f'Total frequent buyer phone numbers: {len(frequent):,}'
    ls['A7'].font = Font(bold=True, name='Arial')
    ls['A8'] = 'Each color group = same customer / same phone number'
    ls['A8'].font = Font(italic=True, name='Arial')
    ls.column_dimensions['A'].width = 40
    ls.column_dimensions['B'].width = 55

    wb.save(output_path)

    print()
    print("  ✓ Done!")
    print(f"  Total rows      : {len(df):,}")
    print(f"  Frequent buyers : {len(frequent):,}")
    print(f"  Saved to        : {output_path}")


if __name__ == '__main__':
    main()

Input  : c:\Users\MPL\Desktop\New folder\input.xls
Output : c:\Users\MPL\Desktop\New folder\Customer Wise Sales Report of 07-06-2026.xlsx

[1/4] Reading file...
        8,927 data rows loaded.
[2/4] Sorting by phone number...
[3/4] Identifying frequent buyers...
        1,430 unique phone numbers with multiple purchases.
[4/4] Writing output.xlsx...

  ✓ Done!
  Total rows      : 8,927
  Frequent buyers : 1,430
  Saved to        : c:\Users\MPL\Desktop\New folder\Customer Wise Sales Report of 07-06-2026.xlsx
